# M2 - Condition classification: train, evaluate, infer (Colab / Kaggle)

**Model:** Keras 3 / TensorFlow classifier `good | moderate | defective` (docs/01-prd.md §10.3).
**Plan:** docs/08-ml-plan.md §2/M2, §3.2, §5 - lightweight MobileNetV3-Small **baseline** vs
stronger EfficientNetV2-S **improved** on the *same* splits / augmentation / seed.

This notebook clones the repo and runs the existing scripts
(`ml/m2_condition_classification/scripts/`) end-to-end:
download data (Roboflow / Kaggle / upload) -> class-map -> 70/15/15 split -> train both
models -> evaluate on the frozen test split -> download the results.

**Before you start**
- Runtime -> Change runtime type -> **T4 GPU** (Colab) / Accelerator **GPU** (Kaggle). CPU works but is slow.
- Pick a license-permitted public damage/condition dataset (docs/08-ml-plan.md §3.2) and document URL + license in the report.
- In cell 1, set `REPO_URL` to your fork/repo.

Run cells in order. Skip the data cells you don't need (3A / 3B / 3C).


In [1]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# 1. SETUP - clone repo, point M2_WORK_ROOT at persistent storage
import os, pathlib

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = os.path.exists("/content")

# TODO: your fork/repo - the one edit that matters for cloning
REPO_URL = "https://github.com/rahulpandiyan/SafeResale.git"
BRANCH   = "main"

WORK_BASE = "/kaggle/working" if IS_KAGGLE else "/content"
REPO_DIR  = os.path.join(WORK_BASE, "SafeResale")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

# Data + runs live here (mirrors ~/safresale-ml/m2 in docs/10-setup-guide.md section 7)
os.environ["M2_WORK_ROOT"] = os.path.join(WORK_BASE, "safresale-ml", "m2")
pathlib.Path(os.environ["M2_WORK_ROOT"]).mkdir(parents=True, exist_ok=True)
print("M2_WORK_ROOT =", os.environ["M2_WORK_ROOT"])

%cd {REPO_DIR}/ml/m2_condition_classification


In [ ]:
# 2. DEPS + GPU CHECK (TF/Keras already ship on Colab and Kaggle)
!pip install -q pyyaml matplotlib roboflow kaggle

import tensorflow as tf, keras
print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", [g.name for g in gpus] or "NONE - CPU fallback (much slower)")


## 3. Provide the dataset - run 3A AND 3B (combined), or 3C (your zip)

The model grades items as `good | moderate | defective`. No single free dataset has all
three grades, so we **combine two license-permitted sources** (configs/class_map.yaml):

| Option | Source | Feeds class | License |
|--------|--------|-------------|---------|
| 3A | Roboflow `damaged-vs-good-packages` (581 imgs, classes `damaged`/`intact`) | good + defective | CC BY 4.0 |
| 3B | Kaggle `dataclusterlabs/cracked-screen-dataset` (cracked screens) | moderate | CC0 (Public Domain) |
| 3C | Your own zip (upload / Google Drive) | any | yours |

**Run both 3A and 3B** - cell 4 merges them into one dataset. 3B is ~900 MB to download
(instant on Kaggle; slower on Colab) and needs a Kaggle account - on Colab it will ask
you to upload `kaggle.json` (https://www.kaggle.com/settings -> 'Create New API Token').

License caveat: many phone/electronics sets on Kaggle are **CC BY-NC** (non-commercial) -
avoid those for a marketplace. Note each dataset's URL + license in the report
(docs/08-ml-plan.md §3.2). Downloads land in `$M2_WORK_ROOT/data/datasets/<name>`.


In [ ]:
# 3A. Roboflow Universe (classification) - damaged-vs-good-packages (CC BY 4.0)
#     Feeds: intact -> good, damaged -> defective. Free key: https://app.roboflow.com/settings/api
import os
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")  # or paste below
ROBOFLOW_WS   = "aadhavs-first-workspace"   # chosen dataset (damaged-vs-good-packages)
ROBOFLOW_PROJ = "damaged-vs-good-packages"  # 581 images, classes: damaged / intact
ROBOFLOW_VER  = 1

if ROBOFLOW_API_KEY and ROBOFLOW_PROJ != "your_project":
    os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
    DATASET_NAME = "condition_packages"
    !python scripts/download_dataset.py --source roboflow --workspace {ROBOFLOW_WS} --project {ROBOFLOW_PROJ} --version {ROBOFLOW_VER} --name {DATASET_NAME}
else:
    print("Skipped - paste ROBOFLOW_API_KEY above, or use 3B / 3C.")


In [ ]:
# 3B. Kaggle cracked-screen set (CC0 Public Domain - commercial OK) -> moderate
#     https://www.kaggle.com/datasets/dataclusterlabs/cracked-screen-dataset
#     Needs kaggle.json (auto on Kaggle; upload on Colab). ~900 MB download.
import os
KAGGLE_DATASET = "dataclusterlabs/cracked-screen-dataset"

if IS_COLAB and not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    from google.colab import files
    print("Upload kaggle.json from https://www.kaggle.com/settings -> 'Create New API Token'")
    files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json

if KAGGLE_DATASET:
    DATASET_NAME = "condition_cracked"
    !python scripts/download_dataset.py --source kaggle --kaggle-dataset {KAGGLE_DATASET} --name {DATASET_NAME} --as-class cracked-screen
else:
    print("Skipped - set KAGGLE_DATASET, or use 3A / 3C.")


In [ ]:
# 3C. Upload your own zip (drag into the Files panel / upload dialog)
#     Expected inside the zip:  <class>/*.jpg   or   {train,valid,test}/<class>/*.jpg
import os, pathlib, zipfile
UPLOAD_ZIP = "condition_data.zip"   # TODO: name of the uploaded zip

if IS_COLAB and not os.path.exists(UPLOAD_ZIP):
    from google.colab import files
    files.upload()

if os.path.exists(UPLOAD_ZIP):
    DATASET_NAME = os.path.splitext(UPLOAD_ZIP)[0]
    dest = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "data" / "datasets" / DATASET_NAME
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOAD_ZIP) as z:
        z.extractall(dest)
    print("extracted", len(list(dest.iterdir())), "top-level entries to", dest)
else:
    print("Skipped - no zip present; upload one or use 3A / 3B.")


## 4. Map labels + build the split

`prepare_dataset.py` builds a **stratified 70/15/15** manifest; the test split is **frozen**
and never used in training. Imbalanced classes get inverse-frequency class weights
(no fabricated labels).

In [ ]:
# 4. COMBINE SOURCES -> one dataset, map labels, then 70/15/15 split
#     (the test split is frozen and never used in training)
import os, pathlib, yaml

# Merge the two sources into a single class-folder dataset (condition_combined):
#   packages: intact -> good, damaged -> defective
#   cracked screens (CC0): -> moderate, capped at 1000 to keep classes balanced
!python scripts/combine_datasets.py --out condition_combined     --src condition_packages/intact     --src condition_packages/damaged     --src condition_cracked/cracked-screen:1000     --overwrite

# Class-map lives in configs/class_map.yaml (committed) - show it for the report
yaml_path = pathlib.Path("configs") / "class_map.yaml"
print("class map:", yaml.safe_load(yaml_path.read_text()))

DATASET_NAME = "condition_combined"
!python scripts/prepare_dataset.py --dataset {DATASET_NAME} --out {DATASET_NAME} --class-map {yaml_path.as_posix()}


In [ ]:
# 5. POINT configs/m2.yaml AT THIS MANIFEST
cfg_path = pathlib.Path("configs/m2.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["dataset"] = DATASET_NAME
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, default_flow_style=False))
print("configs/m2.yaml dataset =", cfg["dataset"])
print("classes =", cfg["classes"], "| models =", [m["tag"] for m in cfg["models"]])


## 6. Train - same protocol for both models (seed 42, identical splits + augmentation)

Two-stage schedule per docs/08-ml-plan.md §5: head on frozen base, then full fine-tune at 1/10 LR.

In [ ]:
# 6A. TRAIN BASELINE - MobileNetV3-Small (lightweight)
!python scripts/train.py --yaml configs/m2.yaml --only baseline-m3small


In [ ]:
# 6B. TRAIN IMPROVED - EfficientNetV2-S (stronger backbone, identical protocol)
!python scripts/train.py --yaml configs/m2.yaml --only improved-effv2s


In [ ]:
# 7. EVALUATE both on the FROZEN test split + per-image latency (docs/01-prd.md 10.3)
#     Writes runs/<tag>/test_metrics.json + confusion matrix PNG/CSV.
WR = os.environ["M2_WORK_ROOT"]
!python scripts/evaluate.py --model {WR}/runs/baseline-m3small/model.keras
!python scripts/evaluate.py --model {WR}/runs/improved-effv2s/model.keras


In [ ]:
# 8. SIDE-BY-SIDE METRICS (measured only - academic-integrity rule)
import json, os, pathlib
for tag in ["baseline-m3small", "improved-effv2s"]:
    p = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "runs" / tag / "test_metrics.json"
    print("===", tag, "===")
    if p.exists():
        m = json.loads(p.read_text())
        print(f"  accuracy={m['accuracy']:.4f}  macro P={m['precision_macro']:.4f}  R={m['recall_macro']:.4f}  F1={m['f1_macro']:.4f}")
        lat = m["latency_per_image_ms"]
        print(f"  latency mean={lat['mean_ms']:.1f}ms p50={lat['p50_ms']:.1f}ms p95={lat['p95_ms']:.1f}ms ({lat['device']})")
        for c, s in m["per_class"].items():
            print(f"    {c:10s} P={s['precision']:.4f} R={s['recall']:.4f} F1={s['f1']:.4f}")
    else:
        print("  missing - run cell 7 first")


In [ ]:
# 9. PACK RESULTS - download runs/ (publish weights via GitHub Releases later)
import os, pathlib, zipfile
runs = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "runs"
out_zip = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "m2_runs.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in runs.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(runs.parent))
print("wrote", out_zip, f"({out_zip.stat().st_size / 1e6:.1f} MB)")
if IS_COLAB:
    from google.colab import files
    files.download(str(out_zip))


## Next steps

- Publish `m2_runs.zip` weights to a **GitHub Release**; teammates run
  `scripts/infer.py --model <downloaded>/runs/<tag>/model.keras --source <img_or_folder>`
  with no dataset required (output matches the `run-vision` condition contract, docs/04-api-contract.md).
- Fill the admin **Models** page `model_metrics` from the `test_metrics.json` files above.
- Set `VISION_PROVIDER=real` + `ML_WEIGHTS_DIR` so `run-vision` uses this model (docs/03-architecture.md §5).
- Document each dataset's URL + license in the report (docs/08-ml-plan.md §3.2).
